In [ ]:
import matplotlib.pyplot as plt
import torch
from torchgeo.trainers import AutoregressionTask

from src.ndvi_dataset import NDVIDataset

In [ ]:
# Modify the following configuration as needed
config = {}
config["ckpt_path"] = None
config["num_past_steps"] = 10
config["num_future_steps"] = 3

In [ ]:
data_dir = "data"

In [ ]:
dataset = NDVIDataset(
    root=data_dir,
    num_past_steps=config["num_past_steps"],
    num_future_steps=config["num_future_steps"],
)

In [ ]:
model = AutoregressionTask.load_from_checkpoint(
    checkpoint_path=config["ckpt_path"],
    teacher_force_prob=None,
)

In [ ]:
time_series = dataset.data.isel(z=0).values

In [ ]:
preds = []
plot_step = 0
for i in range(0, dataset.num_time):
    past_steps = time_series[i : i + dataset.num_past_steps]
    future_steps = time_series[i + dataset.num_past_steps : i + dataset.window_size]

    past_steps_tensor: torch.Tensor = torch.tensor(
        past_steps, dtype=torch.float32
    ).unsqueeze(-1)
    future_steps_tensor: torch.Tensor = torch.tensor(
        future_steps, dtype=torch.float32
    ).unsqueeze(-1)

    mean = past_steps_tensor.mean(dim=0, keepdim=True)
    std = past_steps_tensor.std(dim=0, keepdim=True)
    past_steps_normalized = (past_steps_tensor - mean) / (std + 1e-12)
    future_steps_normalized = (future_steps_tensor - mean) / (std + 1e-12)

    batch = {
        "past_targets": past_steps_normalized.unsqueeze(0),
        "future_targets": future_steps_normalized.unsqueeze(0),
        "mean": mean.unsqueeze(0),
        "std": std.unsqueeze(0),
    }

    pred = model.predict_step(batch, batch_idx=i)
    pred_list = pred.detach().squeeze(0).tolist()
    pred_plot = pred_list[plot_step]
    preds.extend(pred_plot)

In [ ]:
len(preds)

In [ ]:
plt.plot(range(len(time_series)), time_series, marker=".")
plt.plot(
    range(dataset.num_past_steps, len(preds) + dataset.num_past_steps),
    preds,
    marker=".",
)